In [ ]:
dataset = "Xe_hsa_breast_c"  # Set the actual dataset identifier.
import pandas as pd
import numpy as np
import os
import h5py
from scipy.io import mmread
from scipy.sparse import csr_matrix
from glob import glob
import math
import pickle
from matplotlib import pyplot as plt
import matplotlib
import tifffile
from PIL import Image
import subprocess
import shutil 
import gzip
import scipy
import time

# Load Preprocessed Data

Load the aligned H&E, baseline pseudo-spot counts and location table prepared in the preceding Xenium notebooks.

In [ ]:
datadir = "PATH/TO/DATASET"
outdir_root = f"{datadir}/out_of_sample"

In [ ]:
locs_raw = pd.read_csv(
    f'{datadir}/filtered_feature_bc_matrix/spatial/tissue_positions_list_with_header.csv',
    sep=',',
    header=0,
    index_col=0
)
locs_raw

In [ ]:
cnts_raw = pd.read_csv(
    f"{datadir}/cnts.tsv",
    sep='\t',
    header=0,
    index_col=0
)
cnts_raw.index.name = None
cnts_raw

In [ ]:
he_raw = tifffile.imread(f"{datadir}/he-raw.tiff")
plt.imshow(he_raw, cmap='binary')
x_range_max = he_raw.shape[1]
y_range_max = he_raw.shape[0]
print("size of he image")
print("x_range_max: " + str(x_range_max))
print("y_range_max: " + str(y_range_max))

# Load Downloaded Data

Load data from downloaded data in the `data/outs/cell_feature_matrix` folder

In [ ]:
feature_h5_path = "PATH/TO/outs/cell_feature_matrix.h5"
with h5py.File(feature_h5_path, 'r') as f:
    features_group = f['/matrix/features']
    gene_ids = features_group['id'][:]
    gene_names = features_group['name'][:]
    feature_type = features_group['feature_type'][:]
    
    gene_ids = [x.decode('utf-8') if isinstance(x, bytes) else x for x in gene_ids]
    gene_names = [x.decode('utf-8') if isinstance(x, bytes) else x for x in gene_names]
    feature_type = [x.decode('utf-8') if isinstance(x, bytes) else x for x in feature_type]

features_raw_data = pd.DataFrame({'id': gene_ids, 'name': gene_names, 'feature_type': feature_type})
features_raw_data

# Define FOVs

Define the `Size` of Each FOV and the `Distance` between every two FOVs

In [ ]:
size_fov = 1500
# 500 1000 1500
distance_fov = 230
# 500 360 230

file_path = outdir_root + "/fov" + str(size_fov)
if not os.path.exists(file_path):
    os.makedirs(file_path)

min_xs = np.arange((x_range_max - size_fov) % distance_fov / 2, x_range_max - size_fov, distance_fov)
min_ys = np.arange((y_range_max - size_fov) % distance_fov / 2, y_range_max - size_fov, distance_fov)


print("Min X of Each FOV: " + str(min_xs))
print("Min Y of Each FOV: " + str(min_ys))
print("Number of FOV: " + str(len(min_xs) * len(min_ys)))

# Generate Simulate Data

In [ ]:
for min_x_i in range(len(min_xs)):
  for min_y_i in range(len(min_ys)):
    minx = min_xs[min_x_i]
    maxx = minx + size_fov
    miny = min_ys[min_y_i]
    maxy = miny + size_fov
# subset data
    outdir = file_path + "/minx" + str(minx) + "_maxx" + str(maxx) + "_miny" + str(miny) + "_maxy" + str(maxy)
    barcodes = locs_raw[
      (locs_raw['pxl_col_in_fullres'] >= minx) & 
      (locs_raw['pxl_col_in_fullres'] <= maxx) & 
      (locs_raw['pxl_row_in_fullres'] >= miny) & 
      (locs_raw['pxl_row_in_fullres'] < maxy)
    ].index
    if len(barcodes)<30:
        continue
    if not os.path.exists(outdir):
        os.makedirs(outdir)
        os.makedirs(outdir+"/spatial")
        
    locs = locs_raw.loc[barcodes,:]
    exp_data = cnts_raw.loc[barcodes,:]
    locs_sub = locs[['pxl_col_in_fullres','pxl_row_in_fullres']]
    locs_sub.index.name = "spot"
    locs_sub.columns = ["x",'y']
    locs_sub.to_csv(
        f'{outdir}/locs-raw.tsv', 
        sep='\t', 
        index=True,
        header=True
    )
    locs.to_csv(
        f'{outdir}/spatial/tissue_positions_list.csv', 
        sep=',', 
        index=True,
        header=False
    )
    locs.to_csv(
        f'{outdir}/spatial/tissue_positions_list_with_header.csv', 
        sep=',', 
        index=True,
        header=True
    )
    exp_data.to_csv(
        f'{outdir}/cnts.tsv', 
        sep='\t', 
        index=True,
        header=True
    )
# Copy the unchanged full-slide image and metadata for histology-only prediction.
    for filename in ['he-raw.tiff', 'pixel-size-raw.txt', 'radius-raw.txt']:
        shutil.copy2(os.path.join(datadir, filename), os.path.join(outdir, filename))
# generate files
    features_use = pd.DataFrame(exp_data.columns)
    features_use.columns = ['name']
    features_raw_data = features_raw_data.drop_duplicates(subset=['name'], keep='first')
    features = features_raw_data.set_index('name').reindex(exp_data.columns).rename_axis('name').reset_index()
    if features['id'].isna().any():
        raise ValueError("Missing feature metadata for expression-matrix genes")
    features['genome'] = "GRCh38"
    features = features[['id', 'name', 'feature_type', 'genome']]
## save matrix.mtx.gz
    reAnno_count_sparse_mtx = scipy.sparse.csr_matrix(exp_data.T.values)
    with open(os.path.join(outdir, 'matrix.mtx'), "wb") as matrix_file:
        scipy.io.mmwrite(matrix_file, reAnno_count_sparse_mtx)
    with open(os.path.join(outdir,'matrix.mtx'),'rb') as mtx_in:
        with gzip.open(os.path.join(outdir,'matrix.mtx') + '.gz','wb') as mtx_gz:
            shutil.copyfileobj(mtx_in, mtx_gz)
    os.remove(os.path.join(outdir,'matrix.mtx'))
## save barcodes.tsv.gz
    barcodesFile = pd.DataFrame(exp_data.index)
    barcodesFile.to_csv(os.path.join(outdir,"barcodes.tsv.gz"),sep='\t',header =False,index=False)
    ##save features.tsv.gz
    features.to_csv(os.path.join(outdir,"features.tsv.gz"),sep='\t',header =False,index=False)
# generate h5
    expression_data = mmread(f"{outdir}/matrix.mtx.gz").tocsc()
    features = pd.read_csv(f"{outdir}/features.tsv.gz", header=None, sep="\t")
    features.columns = ["id", "name", "feature_type","genome"]
    barcodes = pd.read_csv(f"{outdir}/barcodes.tsv.gz", header=None)
    barcodes.columns = ["barcode"]
    barcode_list = barcodes["barcode"].values
    
    gene_id_list = features["id"].values
    gene_name_list = features["name"].values
    gene_type_list = features["feature_type"].values
    gene_genome_list = features["genome"]
    # generate H5 file
    with h5py.File(f"{outdir}/filtered_feature_bc_matrix.h5", 'w') as f:
    # Add library_ids to attrs
        f.attrs['library_ids'] = np.array([dataset], dtype='S')
    # Add Matrix group
        group = f.create_group('matrix')
    ## Add data
        group.create_dataset('data', data=expression_data.data)
        group.create_dataset('indices', data=expression_data.indices)
        group.create_dataset('indptr', data=expression_data.indptr)
        group.create_dataset('shape', data=expression_data.shape)
        group.create_dataset('barcodes', data=np.array(barcode_list, dtype='S'))
## Add feature group
        feature_group = group.create_group('features')
### Add data
        feature_group.create_dataset('id', data=np.array(gene_id_list, dtype='S'))
        feature_group.create_dataset('name', data=np.array(gene_name_list, dtype='S'))
        feature_group.create_dataset('feature_type', data=np.array(gene_type_list, dtype='S'))
        feature_group.create_dataset('genome', data=np.array(gene_genome_list, dtype='S'))
        feature_group.create_dataset('_all_tag_keys', data=np.array([b'genome'], dtype='S'))
### Add target group
        target_group = feature_group.create_group('target_sets')
        target_group.create_dataset('AddOn Panel', data=np.arange(len(features), dtype=np.int64))
    